# Spurious Currents Animation — σ = 1.0 (Buggy Run)

Animated visualisation of spurious currents developing in a **stationary
circular drop** test (La = 12 000) **before the curvature sign fix**.
The drop should remain at rest, so all velocities are purely numerical
artefacts caused by the sign error in the CSF curvature computation
($\kappa = \nabla\cdot\hat{n}$ instead of $\kappa = -\nabla\cdot\hat{n}$).

| parameter | value |
|-----------|-------|
| ρ | 300 |
| μ | 0.1 |
| σ | 1.0 |
| La = ρσD/μ² | 12 000 |
| R₀ | 0.2 |
| ε (CDI width) | 0.01 |

**This run diverges at t ≈ 0.23.**  The animation captures the full
trajectory from quiescent initial condition through explosive growth of
spurious currents.

See **FINDINGS.md** for the root cause analysis and fix.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.animation as animation
from matplotlib.animation import FuncAnimation, PillowWriter
from mpi4py import MPI
from IPython.display import HTML, Image

from pysemtools.io.ppymech.neksuite import preadnek
from pysemtools.datatypes.msh import Mesh as msh_c
from pysemtools.datatypes.coef import Coef as coef_c
from pysemtools.datatypes.field import Field as field_c

import glob, os

comm = MPI.COMM_WORLD

# Physics parameters (must match spurious_currents.case)
mu = 0.1
sigma = 1.0
rho = 300.0
R0 = 0.2
D = 2 * R0
La = rho * sigma * D / mu**2

# Discover field files — buggy diagnostic run (before sign fix)
data_dir = os.path.join("results", "spurious_diag_sigma1.0")
field_files = sorted(glob.glob(os.path.join(data_dir, "field0.f[0-9]*")))
N_SNAPSHOTS = len(field_files)
print(f"La = {La:.0f}, \u03c3 = {sigma}")
print(f"Found {N_SNAPSHOTS} field files in {data_dir}/")

## Load mesh and build element-local triangulation

In [ ]:
xyz_info = preadnek(field_files[0], comm)
msh = msh_c(comm, data=xyz_info)
coef = coef_c(msh, comm)

# Build element-local triangulation from the structured GLL grid.
n_elem = msh.x.shape[0]
lx = msh.x.shape[3]
ly = msh.x.shape[2]

x_all, y_all, triangles = [], [], []
offset = 0
for e in range(n_elem):
    x_all.append(msh.x[e, 0, :, :].flatten())
    y_all.append(msh.y[e, 0, :, :].flatten())
    for j in range(ly - 1):
        for i in range(lx - 1):
            p0 = offset + j * lx + i
            p1 = offset + j * lx + (i + 1)
            p2 = offset + (j + 1) * lx + i
            p3 = offset + (j + 1) * lx + (i + 1)
            triangles.append([p0, p1, p3])
            triangles.append([p0, p3, p2])
    offset += lx * ly

x = np.concatenate(x_all)
y = np.concatenate(y_all)
triang = tri.Triangulation(x, y, np.array(triangles))
print(f"{n_elem} elements, {len(x)} points, {len(triangles)} triangles")

## Pre-load all snapshots

Read every field file once and cache the z-slice data so the animation
cells can render quickly.

In [ ]:
snapshots = []
for fname in field_files:
    data = preadnek(fname, comm)
    fld = field_c(comm, data=data)
    phi = fld.fields["scal"][0][:, 0, :, :].flatten()
    u = fld.fields["vel"][0][:, 0, :, :].flatten()
    v = fld.fields["vel"][1][:, 0, :, :].flatten()
    vel_mag = np.sqrt(u**2 + v**2)
    Ca_star = mu * np.max(vel_mag) / sigma
    snapshots.append(dict(t=fld.t, phi=phi, u=u, v=v,
                          vel_mag=vel_mag, Ca_star=Ca_star))
    print(f"  {os.path.basename(fname)}: t={fld.t:.6f}, "
          f"u_max={np.max(vel_mag):.4e}, Ca*={Ca_star:.4e}")

# Global velocity range (skip t=0 where everything is zero)
vel_max_global = max(s["vel_mag"].max() for s in snapshots[1:])
print(f"\nGlobal vel_max = {vel_max_global:.4e}")

## Animation 1: Phase field + velocity magnitude (side-by-side)

Left panel: phase field φ with interface contour (φ = 0.5).  
Right panel: velocity magnitude |**u**| on a **fixed** colour scale
so growth of spurious currents is clearly visible.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=100)
fig.subplots_adjust(top=0.82, wspace=0.3)

# Fixed velocity levels for consistent colour mapping
vel_levels = np.linspace(0, vel_max_global, 100)

# Add colorbars once (they persist across frames)
phi_dummy = ax1.tricontourf(triang, snapshots[0]["phi"], levels=100,
                            cmap="RdBu_r", vmin=0, vmax=1)
vel_dummy = ax2.tricontourf(triang, snapshots[0]["vel_mag"],
                            levels=vel_levels, cmap="viridis",
                            vmin=0, vmax=vel_max_global)
fig.colorbar(phi_dummy, ax=ax1, label=r"$\phi$", shrink=0.85)
fig.colorbar(vel_dummy, ax=ax2, label=r"$|\mathbf{u}|$", shrink=0.85)

title = fig.suptitle("", fontsize=12)

def animate_sidebyside(frame):
    s = snapshots[frame]
    for ax in (ax1, ax2):
        for c in ax.collections:
            c.remove()

    ax1.tricontourf(triang, s["phi"], levels=100,
                    cmap="RdBu_r", vmin=0, vmax=1)
    ax1.tricontour(triang, s["phi"], levels=[0.5],
                   colors="w", linewidths=1.2)
    ax1.set_aspect("equal")
    ax1.set_xlabel("x")
    ax1.set_ylabel("y")
    ax1.set_title(r"Phase field $\phi$")

    ax2.tricontourf(triang, s["vel_mag"], levels=vel_levels,
                    cmap="viridis", vmin=0, vmax=vel_max_global)
    ax2.tricontour(triang, s["phi"], levels=[0.5],
                   colors="w", linewidths=1.2, linestyles="--")
    ax2.set_aspect("equal")
    ax2.set_xlabel("x")
    ax2.set_title(r"Velocity magnitude $|\mathbf{u}|$")

    title.set_text(
        f"Spurious currents \u2014 \u03c3={sigma}, La={La:.0f}\n"
        f"t = {s['t']:.4f}   Ca* = {s['Ca_star']:.4e}")
    return []

anim1 = FuncAnimation(fig, animate_sidebyside,
                      frames=N_SNAPSHOTS, interval=600, blit=False)
anim1.save("sigma_1.0_sidebyside.gif",
           writer=PillowWriter(fps=2), dpi=100)
plt.close(fig)
print("Saved: sigma_1.0_sidebyside.gif")
Image(filename="sigma_1.0_sidebyside.gif")

## Animation 2: Log-scale velocity with quiver arrows near interface

Uses a **logarithmic** colour scale for velocity magnitude so that both
the early weak currents (Ca* ~ 10⁻⁴) and the final explosive growth
(Ca* ~ 1) are visible in the same animation.

In [ ]:
from matplotlib.colors import LogNorm
import matplotlib.colors as mcolors

# Velocity range for log scale (skip t=0)
vel_min_nonzero = min(s["vel_mag"][s["vel_mag"] > 0].min()
                      for s in snapshots[1:])
log_vmin = 10**np.floor(np.log10(vel_min_nonzero))
log_vmax = 10**np.ceil(np.log10(vel_max_global))

fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
fig.subplots_adjust(top=0.85)

# Log-spaced levels for tricontourf
log_levels = np.logspace(np.log10(log_vmin), np.log10(log_vmax), 60)

title = fig.suptitle("", fontsize=11)

def animate_log(frame):
    s = snapshots[frame]
    ax.clear()

    # Clamp velocity for log scale (avoid log(0))
    vel_clamped = np.clip(s["vel_mag"], log_vmin, None)

    c = ax.tricontourf(triang, vel_clamped, levels=log_levels,
                       norm=LogNorm(vmin=log_vmin, vmax=log_vmax),
                       cmap="inferno")

    # Interface
    ax.tricontour(triang, s["phi"], levels=[0.5],
                  colors="cyan", linewidths=1.5)

    # Quiver arrows near the interface
    near = (s["phi"] > 0.05) & (s["phi"] < 0.95)
    idx = np.where(near)[0][::4]
    if len(idx) > 0:
        xq, yq = x[idx], y[idx]
        magq = s["vel_mag"][idx]
        u_norm = s["u"][idx] / (magq + 1e-30)
        v_norm = s["v"][idx] / (magq + 1e-30)
        ax.quiver(xq, yq, u_norm, v_norm,
                  color="cyan", alpha=0.6,
                  scale=45, width=0.003, pivot="mid",
                  headwidth=4, headlength=5)

    ax.set_xlim(0.15, 0.85)
    ax.set_ylim(0.15, 0.85)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    title.set_text(
        f"Spurious currents (log scale) \u2014 \u03c3={sigma}, La={La:.0f}\n"
        f"t = {s['t']:.4f}   Ca* = {s['Ca_star']:.4e}")
    return []

anim2 = FuncAnimation(fig, animate_log,
                      frames=N_SNAPSHOTS, interval=600, blit=False)
anim2.save("sigma_1.0_logscale.gif",
           writer=PillowWriter(fps=2), dpi=100)
plt.close(fig)
print("Saved: sigma_1.0_logscale.gif")
Image(filename="sigma_1.0_logscale.gif")

## Animation 3: Zoomed interface with per-frame normalised velocity

Each frame rescales the colour bar to its own velocity range so that the
spatial **structure** of the spurious currents is visible at every time
(useful early on when they are O(10⁻⁴) and invisible on the global scale).

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6), dpi=100)
fig.subplots_adjust(top=0.85, right=0.82)
cax = fig.add_axes([0.85, 0.15, 0.03, 0.65])
title = fig.suptitle("", fontsize=11)

def animate_perframe(frame):
    s = snapshots[frame]
    ax.clear()
    cax.clear()

    vmax_frame = s["vel_mag"].max() if s["vel_mag"].max() > 0 else 1e-10

    c = ax.tricontourf(triang, s["vel_mag"], levels=100,
                       cmap="viridis", vmin=0, vmax=vmax_frame)

    # Interface contour
    ax.tricontour(triang, s["phi"], levels=[0.5],
                  colors="white", linewidths=2)

    # Quiver near interface
    near = (s["phi"] > 0.05) & (s["phi"] < 0.95)
    idx = np.where(near)[0][::3]
    if len(idx) > 0:
        xq, yq = x[idx], y[idx]
        magq = s["vel_mag"][idx]
        u_norm = s["u"][idx] / (magq + 1e-30)
        v_norm = s["v"][idx] / (magq + 1e-30)
        ax.quiver(xq, yq, u_norm, v_norm, magq,
                  cmap="autumn", clim=[0, vmax_frame],
                  scale=40, width=0.004, pivot="mid")

    fig.colorbar(c, cax=cax, label=r"$|\mathbf{u}|$")

    ax.set_xlim(0.2, 0.8)
    ax.set_ylim(0.2, 0.8)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    title.set_text(
        f"Per-frame normalised \u2014 \u03c3={sigma}, La={La:.0f}\n"
        f"t = {s['t']:.4f}   Ca* = {s['Ca_star']:.4e}   "
        f"u_max = {vmax_frame:.3e}")
    return []

anim3 = FuncAnimation(fig, animate_perframe,
                      frames=N_SNAPSHOTS, interval=600, blit=False)
anim3.save("sigma_1.0_perframe.gif",
           writer=PillowWriter(fps=2), dpi=100)
plt.close(fig)
print("Saved: sigma_1.0_perframe.gif")
Image(filename="sigma_1.0_perframe.gif")

## Static frame gallery

All snapshots side-by-side for inclusion in a report/paper.  Uses the
global velocity scale so growth is directly comparable.

In [ ]:
n_cols = 5
n_rows = (N_SNAPSHOTS + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(4 * n_cols, 4.2 * n_rows), dpi=120)
axes = axes.flatten()

for i, s in enumerate(snapshots):
    ax = axes[i]
    ax.tricontourf(triang, s["vel_mag"], levels=vel_levels,
                   cmap="viridis", vmin=0, vmax=vel_max_global)
    ax.tricontour(triang, s["phi"], levels=[0.5],
                  colors="w", linewidths=1.5)
    ax.set_aspect("equal")
    ax.set_xlim(0.15, 0.85)
    ax.set_ylim(0.15, 0.85)
    ax.set_title(f"t={s['t']:.4f}\nCa*={s['Ca_star']:.2e}", fontsize=9)
    ax.tick_params(labelsize=7)

for i in range(N_SNAPSHOTS, len(axes)):
    axes[i].set_visible(False)

fig.suptitle(
    f"Spurious currents evolution \u2014 \u03c3={sigma}, La={La:.0f}\n"
    f"(fixed colour scale, white contour = \u03c6 = 0.5)",
    y=1.02)
plt.tight_layout()
plt.savefig("sigma_1.0_gallery.png", dpi=150, bbox_inches="tight")
plt.show()

## Ca*(t) from ekin.csv

Full time history of the spurious capillary number from the Dardel run,
with the field-file snapshot times marked.

In [ ]:
ekin_data = np.genfromtxt(os.path.join(data_dir, "ekin.csv"),
                          delimiter=",", comments="#")
t_csv = ekin_data[:, 0]
u_max_csv = ekin_data[:, 3]
Ca_csv = mu * u_max_csv / sigma

snap_times = np.array([s["t"] for s in snapshots])
snap_Ca = np.array([s["Ca_star"] for s in snapshots])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5), dpi=120)

# Linear scale
ax1.plot(t_csv, Ca_csv, "-", lw=1, label="ekin.csv")
ax1.plot(snap_times, snap_Ca, "ro", markersize=5, label="field snapshots")
ax1.set_xlabel("t")
ax1.set_ylabel(r"$Ca^* = \mu\, u_{\max} / \sigma$")
ax1.set_title("Linear scale")
ax1.legend()
ax1.grid(True)

# Log scale
mask = Ca_csv > 0
ax2.semilogy(t_csv[mask], Ca_csv[mask], "-", lw=1, label="ekin.csv")
mask_snap = snap_Ca > 0
ax2.semilogy(snap_times[mask_snap], snap_Ca[mask_snap], "ro",
             markersize=5, label="field snapshots")
ax2.set_xlabel("t")
ax2.set_ylabel(r"$Ca^*$")
ax2.set_title("Log scale (reveals exponential growth)")
ax2.legend()
ax2.grid(True, which="both", alpha=0.4)

fig.suptitle(
    f"Spurious Ca* vs time (buggy run) \u2014 \u03c3={sigma}, La={La:.0f}",
    y=1.02)
plt.tight_layout()
plt.savefig("sigma_1.0_Ca_star.png", dpi=150, bbox_inches="tight")
plt.show()